# 한국경제 기사 크롤링

In [93]:
# 필요 모듈 import
from selenium import webdriver
from selenium.webdriver.support.ui import Select
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import ElementClickInterceptedException, NoSuchElementException, StaleElementReferenceException
import csv, time, re
import pandas as pd
from datetime import datetime


In [94]:
# Chrome headless 설정
options = Options()
options.add_argument("--headless")
service = Service(executable_path='../chromedriver-mac-arm64/chromedriver')
driver = webdriver.Chrome(service=service, options=options)




In [95]:
# 초기 페이지 접속
#for category in range(len(news_cat)):

driver.get('https://www.hankyung.com/economy/economic-policy?page=1')
wait = WebDriverWait(driver, 5)

MAX_PAGE = 1
article_links = []

for i in range(MAX_PAGE):
    try:
        print(f"{i+1}번째 페이지 링크 수집중...")

        # 드롭다운 셀렉터를 매번 새로 불러오기 (중요!)
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, '#contents > div.select-paging > div.page-select.txt-num > div > select')))
        select_element = driver.find_element(By.CSS_SELECTOR, '#contents > div.select-paging > div.page-select.txt-num > div > select')
        select = Select(select_element)

        # 드롭다운에서 i+1 페이지 선택
        select.select_by_value(str(i + 1))
        time.sleep(2)

        # 기사 a태그 대기 및 수집
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, '#contents > ul a')))
        a_tags = driver.find_elements(By.CSS_SELECTOR, '#contents > ul a')
        for a in a_tags:
            href = a.get_attribute('href')
            if href:
                article_links.append(href)

    except (NoSuchElementException, StaleElementReferenceException) as e:
        print(f"[예외 발생] 페이지 {i+1} 수집 중 오류: {e}")
        break

# 중복 제거
article_list = list(set(article_links))
print(f"\n총 {len(article_list)}개의 기사 링크를 수집했습니다.")

1번째 페이지 링크 수집중...

총 20개의 기사 링크를 수집했습니다.


In [96]:
article = {}

for i in range(len(article_list)):
    driver.get(article_list[i])
    time.sleep(2)
    try:
        print(f"{i+1}번째 기사 크콜링중...")
        # 기사 본문 로딩 대기
        article_element = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, '#articletxt')))
    
        # 텍스트 수집
        article_text = article_element.text
        article_text.strip()
    except Exception as e:
        print(f"[예외 발생] 기사 본문 수집 실패: {e}")
    finally:
        # 딕셔너리에 URL을 키로, 본문을 값으로 저장한다.
        article[article_list[i]] = article_text

print(f"총{len(article)}개의 기사 수집 완료")

1번째 기사 크콜링중...
2번째 기사 크콜링중...
3번째 기사 크콜링중...
4번째 기사 크콜링중...
5번째 기사 크콜링중...
6번째 기사 크콜링중...
7번째 기사 크콜링중...
8번째 기사 크콜링중...
9번째 기사 크콜링중...
10번째 기사 크콜링중...
11번째 기사 크콜링중...
12번째 기사 크콜링중...
13번째 기사 크콜링중...
14번째 기사 크콜링중...
15번째 기사 크콜링중...
16번째 기사 크콜링중...
17번째 기사 크콜링중...
18번째 기사 크콜링중...
19번째 기사 크콜링중...
20번째 기사 크콜링중...
총20개의 기사 수집 완료


In [97]:
# 크롤링 내용 csv로 저장

with open('../data/raw/news/korea_economy.csv', 'w', newline="", encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['URL', 'content'])

    for url, content in article.items():
        writer.writerow([url, content])
    

# 뉴스기사 전처리

## 결측치 처리 및 월별로 정렬

In [98]:
korea = pd.read_csv('../data/raw/news/korea_economy.csv')
# 결측치 기사의 수가 적어서 그냥 다 삭제하기로 결정
korea.dropna(inplace=True)

In [99]:
# 날짜 추출 함수 정의
def extract_date(url):
    match = re.search(r'/article/(\d{8})', url)
    if match:
        return datetime.strptime(match.group(1), "%Y%m%d").date()
    return None

# 날짜 컬럼 추가
korea['date'] = korea['URL'].apply(extract_date)

In [100]:
# 날짜 순으로 정렬후 URL 삭제
korea.sort_values(by='date',ascending=False, inplace=True)
korea.drop('URL', axis=1, inplace=True)
# 컬럼 순서 변경
korea = korea[['date','content']]
# 인덱스 번호 초기화
korea = korea.reset_index(drop=True)


In [103]:
korea.to_csv('../data/interim/news/interim_korea_economy.csv', index=False)